In [ ]:
# scheduler.py
from apscheduler.schedulers.blocking import BlockingScheduler
from zoneinfo import ZoneInfo
from datetime import datetime, timedelta
from holidays_utils import market_status, market_holidays
from get_symbols import get_active_symbols, update_symbols
from intraday import update_intraday_for_symbols
from daily import update_daily_data
from archive_job import archive_old_intraday
from time_utils import now_ny, in_work_window
import time as _time

NY = ZoneInfo("America/New_York")
sched = BlockingScheduler(timezone=NY)

# 你想要的工作時段（NY 時區）
WORK_START = (9,45)   # 09:45
WORK_END = (16,15)    # 16:15

# 每分鐘 1m job（job 內會先檢查 market_status 與工作時段）
@sched.scheduled_job('cron', minute='*')
def job_intraday_1m():
    try:
        status = market_status()
    except Exception as e:
        print("market_status error:", e)
        return
    if not status.get("marketOpen", False):
        return
    if not in_work_window(start_hm=WORK_START, end_hm=WORK_END):
        return
    symbols = get_active_symbols()
    update_intraday_for_symbols(symbols, "1m")

# 每 5 分鐘 5m job
@sched.scheduled_job('cron', minute='*/5')
def job_intraday_5m():
    try:
        status = market_status()
    except Exception as e:
        print("market_status error:", e)
        return
    if not status.get("marketOpen", False):
        return
    if not in_work_window(start_hm=WORK_START, end_hm=WORK_END):
        return
    symbols = get_active_symbols()
    update_intraday_for_symbols(symbols, "5m")

# daily update：market close + buffer（16:20 NY time）
@sched.scheduled_job('cron', hour=17, minute=20)
def job_daily_update():
    try:
        status = market_status()
    except Exception as e:
        print("market_status error:", e)
        return
    if status.get("marketOpen", True):
        return
    update_daily_data()

# archive job：每日凌晨 NY time 02:00
@sched.scheduled_job('cron', hour=9, minute=20)
def job_archive():
    archive_old_intraday()

# symbols update：每日凌晨 NY time 02:10
@sched.scheduled_job('cron', hour=, minute=00)
def job_update_symbols():
    update_symbols()

if __name__ == "__main__":
    print("Scheduler starting (NY time)...")
    sched.start()
